# 05 · Planner (LLM)

**Role of this layer.** The planner turns a request into a plan, which is inert
JSON. It is one of only two places an LLM is used (the other is the narrator).

* The LLM sees the **generated** capability summary, dataset **metadata** and
  rules, never data values. The request is wrapped in `<user_request>` tags and
  treated as data, not as instructions.
* Output arrives through a `submit_plan` tool whose input schema is the plan
  schema; it is then parsed strictly (V01) and validated like any other plan.
* Modes: **live** (call the API), **cached** (reuse a cached plan for an
  identical request, prompt, model and registry), **mock** (canned plans for the
  five example workflows; no API key needed).

This notebook uses live mode if the configured provider's API key is set
(`QUANTLAB_LLM_PROVIDER`, default `openai`, with `OPENAI_API_KEY`), otherwise mock.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [4]:
from quantlab.data import load_catalog
from quantlab.mock_scenarios import SCENARIOS_BY_ID
from quantlab.plan import describe_plan, plan_tool_schema
from quantlab.planner import PROMPT_VERSION, build_system_prompt, cache_key, generate_plan
from quantlab.validator import fill_defaults, validate_plan

catalog = load_catalog(settings.catalog_path)
mode = "live" if settings.has_api_key else "mock"
print(
    "mode:",
    mode,
    "| provider:",
    settings.llm_provider,
    "| model:",
    settings.planner_model,
    "| prompt:",
    PROMPT_VERSION,
)

mode: live | provider: openai | model: gpt-6-luna | prompt: planner-v8


The start of the system prompt (the registry summary and dataset metadata follow):

In [5]:
prompt = build_system_prompt(catalog, ["NASDAQCOM"])
print(prompt[:1800])
print(f"... ({len(prompt):,} characters in total)")

You are the planner of QuantLab, a quantitative analysis assistant used by a bank's markets and risk teams. You turn a user's concrete analytical question into a PLAN: an ordered list of steps, each calling one capability from the approved registry below. If the request is not such a question, you decline it.

You never write or run code. You only choose registered capabilities and fill in their parameters. A deterministic validator checks your plan, a human reviews it, and a deterministic engine runs it.

Rules:
1. Plan only when the request states a concrete analytical question: which series (named in the request, or a selected dataset) and what to compute, compare or test about it. If it does not (for example "surprise me", "analyse the market" or "what is the most interesting analysis?"), set cannot_fulfill=true. In refusal_reason, say that the request has no concrete question, say in one sentence what QuantLab can compute, and suggest one or two concrete requests that name a serie

In [6]:
schema = plan_tool_schema()
print("tool schema fields:", list(schema["properties"]))
print("step fields:", list(schema["properties"]["steps"]["items"]["properties"]))

tool schema fields: ['dataset_ids', 'steps', 'cannot_fulfill', 'refusal_reason', 'planner_notes']
step fields: ['id', 'capability', 'inputs', 'params', 'rationale']


Planning scenario 1, titled "Is the Swiss franc a safe haven?":

In [7]:
scenario = SCENARIOS_BY_ID["chf_safe_haven_success"]
result = generate_plan(scenario.request, list(scenario.dataset_ids), mode, settings, catalog)
print(
    "provider:",
    result.provider,
    "| served by:",
    result.served_model,
    "| from cache:",
    result.from_cache,
    "| latency ms:",
    result.latency_ms,
)
print(result.error or describe_plan(result.plan))
if result.plan:
    print("validation:", validate_plan(fill_defaults(result.plan), catalog).codes())

provider: openai | served by: gpt-6-luna | from cache: False | latency ms: 8097.4
s1 load_series(dataset_id=VIXCLS)
s2 resample(series=s1, freq=W, how=last)
s3 flag_regime(series=s2, quantile=0.9, method=expanding, min_periods=52, direction=above)
s4 load_series(dataset_id=DEXSZUS)
s5 resample(series=s4, freq=W, how=last)
s6 forward_return(series=s5, method=simple)
s7 hac_mean_difference(outcome=s6, flags=s3)
validation: ['V10']


A request for a model (scenario 3) never runs: there is no capability for model
fitting or prediction, so the planner refuses (P01) and says what is possible.

In [8]:
prediction = SCENARIOS_BY_ID["arima_forecast_block"]
result_c = generate_plan(prediction.request, list(prediction.dataset_ids), mode, settings, catalog)
plan_c = result_c.plan
if plan_c is None:
    print("no plan:", result_c.error)
elif plan_c.cannot_fulfill:
    print("refused:", plan_c.refusal_reason)
else:
    print(describe_plan(plan_c))
    print("validation:", validate_plan(fill_defaults(plan_c), catalog).codes())

refused: QuantLab does not have an ARIMA capability or forecasting capability, so it cannot fit the requested model or forecast the next 12 weeks.


A prompt injection (scenario 4). In mock mode its planner is simulated and misbehaves: it follows the
injection and proposes an unregistered capability. The plan is only a proposal: the validator
blocks it (V02), and nothing runs (a real planner is expected to refuse, as in scenario 3):

In [9]:
from quantlab.validator import fill_defaults, validate_plan

injection = SCENARIOS_BY_ID["prompt_injection_unknown_capability"]
result_d = generate_plan(injection.request, list(injection.dataset_ids), mode, settings, catalog)
plan_d = result_d.plan
print("proposed steps:", [step.capability for step in plan_d.steps] if plan_d else result_d.error)
if plan_d is not None:
    report_d = validate_plan(fill_defaults(plan_d), catalog)
    print("validator:", [issue.issue_id for issue in report_d.issues])

proposed steps: []
validator: ['P01:plan']


The plan cache key covers everything that could change the answer: request,
datasets, model, prompt version and registry version.

In [10]:
print(cache_key(scenario.request, list(scenario.dataset_ids), settings.planner_model))
print(cache_key(scenario.request, list(scenario.dataset_ids), "another-model"))

48d75a0ef621a9106ef55bde1a88599c61dbc51733b5a1b7b4d78787c7f380c2
d064a1db1ce1bd4454befaa59ed1dbbca7eaee8fe3733f1ad24cdb2d9599a595
